# Artikkeloversikt

Henter strukturert data fra SQLite (elementer, sammendrag, evalueringstriplets) og eksporterer til Excel.

In [ ]:
import sqlite3, os
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from IPython.display import display

repo_rot = Path("__file__").resolve().parent.parent
load_dotenv(repo_rot / ".env")
DB_STI = repo_rot / os.getenv("DATABASE_STI", "data/monitor.db")

print(f"Database : {DB_STI}  (finnes: {DB_STI.exists()})")

In [ ]:
SQL = """
SELECT
    e.id            AS element_id,
    k.navn          AS kilde,
    e.tittel,
    e.publisert,
    e.url,
    s.tekst         AS sammendrag,
    s.prompt_versjon,
    t.godkjent,
    t.kommentar     AS ekspert_kommentar,
    t.tidsstempel   AS vurdert
FROM elementer e
JOIN kilder k ON e.kilde_id = k.id
LEFT JOIN sammendrag s ON s.element_id = e.id
LEFT JOIN evalueringstriplets t
    ON t.element_id = e.id AND t.komponent = 'sammendrag'
WHERE e.dead_letter = 0
ORDER BY e.publisert DESC
"""

with sqlite3.connect(DB_STI) as conn:
    df = pd.read_sql_query(SQL, conn)

df["godkjent"] = df["godkjent"].map({1: "Godkjent", 0: "Avvist"}).fillna("Ikke vurdert")

print(f"Rader: {len(df)}  |  Kolonner: {list(df.columns)}")

In [ ]:
pd.set_option("display.max_colwidth", None)

def klipp(tekst, maks=200):
    if pd.isna(tekst):
        return ""
    return tekst[:maks] + "…" if len(tekst) > maks else tekst

vis = df.copy()
vis["sammendrag"] = vis["sammendrag"].apply(klipp)

display(
    vis.style
    .set_properties(**{"text-align": "left", "white-space": "pre-wrap"})
    .set_table_styles([{"selector": "th", "props": [("text-align", "left")]}])
)

In [ ]:
from datetime import date

eksport_mappe = repo_rot / "data" / "eksport"
eksport_mappe.mkdir(parents=True, exist_ok=True)

filnavn = eksport_mappe / f"artikkeloversikt_{date.today()}.xlsx"
df.to_excel(filnavn, index=False, engine="openpyxl")

print(f"Lagret: {filnavn.resolve()}")